# Protein Design Agent Using Strands

This notebook demonstrates how to create a protein design agent using the open-source Strands Agents framework.

## Learning Objectives 

- Build a protein design agent using the Strands framework
- Learn how to execute AWS HealthOmics workflows
- Monitor workflows and analyze results


## 1. Environment Setup

#### Install Strands agents and required dependencies

In [ ]:
%pip install strands-agents strands-agents-tools --quiet

#### Verify latest version of boto3 shown below
Verify that the boto3 version shown below is **1.37.1** or higher.

In [ ]:
%pip show boto3

#### Import required libraries

In [ ]:
import boto3
import json
import time
import uuid
import os
from strands import Agent, tool
from strands.models import BedrockModel
from urllib.parse import urlparse

# Get AWS account information
sts_client = boto3.client('sts')
account_id = sts_client.get_caller_identity()['Account']



In [ ]:
session = boto3.Session(
    #aws_access_key_id='your_access_key',
    #aws_secret_access_key='your_secret_key',
    #aws_session_token='your_session_token',  # If using temporary credentials
    #region_name='us-west-2',
    #profile_name='your-profile'  # Optional: Use a specific profile
)
region = session.region_name

### Prerequisites

Run the notebook environment setup in [00-setup_environment.ipynb](00-setup_environment.ipynb).

Also ensure you have deployed the protein design CloudFormation stack from `stacks/protein_design_stack.yaml`.

<img src="https://github.com/hsr87/strands-agents-for-life-science/blob/main/notebook/images/protein-stack-completion.png?raw=true" width="400" alt="CloudFormation Checking">


### Set up AWS clients and configuration
Define clients for AWS services to use in tools.

In [ ]:
STACK_NAME = 'protein-design-stack'  # Your CloudFormation stack name
DEFAULT_WORKFLOW_ID = None  # Retrieved from stack outputs
DEFAULT_ROLE_ARN = None     # Retrieved from stack outputs
DEFAULT_S3_BUCKET = f"{STACK_NAME}-{account_id}-{region}"    # Predefined format in CloudFormation.

# Set environment variables for use in tools
os.environ['STACK_NAME'] = STACK_NAME

print(f"Region: {region}")
print(f"Account ID: {account_id}")
print(f"Stack Name: {STACK_NAME}")

### Get CloudFormation stack outputs

In [ ]:
# Get CloudFormation stack outputs
cf_client = boto3.client('cloudformation')

try:
    response = cf_client.describe_stacks(StackName=STACK_NAME)
    stack = response['Stacks'][0]

    # Retrieve from outputs
    outputs = stack.get('Outputs', [])
    for output in outputs:
        key = output['OutputKey']
        value = output['OutputValue']
        if key == 'WorkflowId':
            DEFAULT_WORKFLOW_ID = value
        elif key == 'WorkflowExecutionRoleArn':
            DEFAULT_ROLE_ARN = value
            
    parameters = stack.get('Parameters', [])

    
    print(f"Workflow ID: {DEFAULT_WORKFLOW_ID}")
    print(f"Role ARN: {DEFAULT_ROLE_ARN}")
    print(f"S3 Bucket: {DEFAULT_S3_BUCKET}")
    
    # Configure tools with stack information
    from utils.protein_design_tools import set_stack_config
    set_stack_config(
        stack_name=STACK_NAME,
        workflow_id=DEFAULT_WORKFLOW_ID,
        role_arn=DEFAULT_ROLE_ARN,
        s3_bucket=DEFAULT_S3_BUCKET
    )
    
except Exception as e:
    print(f"Error retrieving stack outputs: {e}")
    print("Please update configuration variables manually")

## 2. Create Strands Agent
This section creates an agent using the Strands framework

### Define agent configuration and instructions

In [ ]:
protein_agent_name = 'Protein-Design-Agent-Strands'
protein_agent_description = "Protein design and optimization agent using AWS HealthOmics workflows via the Strands framework"
protein_agent_instruction = """
You are a protein design AI specialized in helping researchers optimize protein sequences using directed evolution algorithms.
You can trigger AWS HealthOmics workflows to perform protein sequence optimization and monitor progress.

Your capabilities include:
1. Starting protein design optimization workflows with custom parameters
2. Monitoring status of running workflows
3. Retrieving and analyzing results from completed optimizations

When working with protein sequences:
- Validate that sequences contain only valid amino acid characters
- Provide clear explanations of the optimization process
- Help users understand results and their implications

Always be helpful and provide detailed information about the protein design process.
"""

#### Define tools for Strands agent

These tools will invoke and monitor AWS HealthOmics workflows for protein design tasks.

In [ ]:
# Get updated tools matching Bedrock agent action groups
from utils.protein_design_tools import trigger_aho_workflow, monitor_aho_workflow

print("Tools successfully imported:")
print(f"- trigger_aho_workflow: {trigger_aho_workflow.__doc__.split('Args:')[0].strip()}")
print(f"- monitor_aho_workflow: {monitor_aho_workflow.__doc__.split('Args:')[0].strip()}")

#### Create Strands agent

In [ ]:
# Create Bedrock model
model = BedrockModel(
    model_id="us.anthropic.claude-3-7-sonnet-20250219-v1:0",
    boto_session=session
)

protein_agent = Agent(
    system_prompt=protein_agent_instruction,
    model=model,
    tools=[trigger_aho_workflow, monitor_aho_workflow]
)

print(f"Created protein design agent with the Strands framework")

# Test Agent
Let's test the protein design agent with some example queries.

#### (Optional) Debugging 

You can use StrandsAgents' logging module for debugging if needed.

In [ ]:
#https://strandsagents.com/latest/documentation/docs/user-guide/observability-evaluation/logs/
#import logging

# Configure root strands logger
#logging.getLogger("strands").setLevel(logging.DEBUG)

# Add handler to view logs
#logging.basicConfig(
#    format="%(levelname)s | %(name)s | %(message)s", 
#    handlers=[logging.StreamHandler()]
#)

#### Test 1: Start Protein Optimization

In [ ]:
# Test starting protein optimization
test_sequence = "EVQLVETGGGLVQPGGSLRLSCAASGFTLNSYGISWVRQAPGKGPEWVSVIYSDGRRTFYGDSVKGRFTISRDTSTNTVYLQMNSLRVEDTAVYYCAKGRAAGTFDSWGQGTLVTVSS"

query = f"Please optimize this protein sequence: {test_sequence}"

print("Query:", query)

try:
    # Execute agent
    print("\nResponse:")
    protein_agent(query)
except Exception as e:
    print(f"Error during agent execution: {e}")
    import traceback
    traceback.print_exc()

#### Test 2: Monitor Workflow Status

You can also enter the HealthOmics console via the Run menu to check the status of the Run ID you executed above. (In this example image, the Run ID is 2951139, which will be different for each user)

<img src="https://github.com/hsr87/strands-agents-for-life-science/blob/main/notebook/images/healthomics.png?raw=true" width="400" alt="HealthOmics Run">


In [ ]:
# Test workflow monitoring (use run ID from previous test)
# Replace 'YOUR_RUN_ID' with the actual run ID from the previous test
test_run_id = "{test_run_id}"  # Update with actual run ID

query = f"Please check the status of workflow run {test_run_id}"

print("Query:", query)
print("\nResponse:")
response = protein_agent(query)

#### Test 3: Advanced Optimization with Custom Parameters

In [ ]:
# Test with custom parameters
query = "Please run a protein optimization on the sequence ACDEFGHIKLMNPQRSTVWY with 20 parallel chains and 200 steps"

print("Query:", query)
print("\nResponse:")
response = protein_agent(query)

In [ ]:
# Metrics: https://strandsagents.com/latest/documentation/docs/user-guide/observability-evaluation/metrics/

# Access metrics through AgentResult
print(f"Total tokens: {response.metrics.accumulated_usage['totalTokens']}")
print(f"Execution time: {sum(response.metrics.cycle_durations):.2f} seconds")
print(f"Tools used: {list(response.metrics.tool_metrics.keys())}")

## Conclusion

In this notebook, we successfully implemented a protein design agent based on AWS HealthOmics workflows using the Strands Agents framework.

### Key Implementation Details:

#### 1. Protein Design Agent Development
- **Strands Framework**: Implemented protein design specialized AI using open-source agent framework
- **AWS HealthOmics Integration**: Cloud-based life sciences workflow execution and management

#### 2. Core Tool Implementation
- **trigger_aho_workflow**: Start protein optimization workflow with custom parameters
- **monitor_aho_workflow**: Real-time monitoring of running workflow status
- **CloudFormation Integration**: Consistent environment configuration through infrastructure automation

#### 3. Agent Functionality
- Amino acid sequence validation
- Custom optimization parameter configuration (number of parallel chains, optimization steps, etc.)
- Workflow execution status tracking and result analysis
- Detailed explanation of protein design process

#### 4. AWS Service Integration
- **AWS HealthOmics**: Life sciences workflow execution platform
- **Amazon Bedrock**: Natural language processing with Claude 3.7 Sonnet model
- **Amazon S3**: Workflow input/output data storage
- **AWS IAM**: Security role and permission management

### Practical Use Examples:

**Protein Optimization Workflow:**

User Input: "Please optimize the sequence EVQLVETGGGLVQPGGSLRLSCAASGFTLNSYGISWVRQAPGKGPEWVSVIYSDGRRTFYGDSVKGRFTISRDTSTNTVYLQMNSLRVEDTAVYYCAKGRAAGTFDSWGQGTLVTVSS"
↓ Agent Processing
• Sequence validation
• Trigger HealthOmics workflow
• Monitor execution status
• Analyze and interpret results

**Advanced Parameter Configuration:**

"Please run a protein optimization with 20 parallel chains and 200 steps"
→ Adjust optimization performance with custom parameters

### Technical Features:

#### 1. Scalability
- Handle large-scale protein optimization tasks in the cloud
- Efficient computing resource utilization through parallel processing (using managed service AWS HealthOmics)

#### 2. Usability
- Easy execution of complex life science workflows through natural language interface
- Transparent process tracking through real-time monitoring

#### 3. Integration
- Complete integration with AWS ecosystem
- Infrastructure as code through CloudFormation

### Application Areas:
- **Drug Development**: Optimization of therapeutic proteins
- **Enzyme Engineering**: Performance improvement of industrial enzymes
- **Antibody Design**: Specificity and affinity enhancement
- **Vaccine Development**: Immunogenicity optimization

Through this implementation, life science researchers can now perform advanced protein design tasks through natural language conversation with AI agents without complex computational biology knowledge. The combination of **Strands Agents framework** and **AWS HealthOmics** has greatly improved accessibility in the protein design field.
